In [ ]:
# %% [markdown]
# BEACON — Preprocessing Steps 5-10
# Continues from beacon_preprocess_steps1to4.py's output (CLEAN_DIR).
# Requires: pip install scikit-learn imbalanced-learn
#
# Order matters here — this follows the exact sequence you specified:
#   5. Grouped stratified split (BEFORE any resampling)
#   6. Log-transform heavy-tailed features
#   7. Min-Max scale (fit on train only — see note in that cell)
#   8. Prune correlated features (computed on train only)
#   9. Network: light imbalance handling (moderate ~5:1)
#  10. Memory: cluster-based SMOTE for Exploit only (severe ~10-16:1),
#      light handling for the other 8 categories

# %%
import os
import numpy as np
import pandas as pd
import polars as pl
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

CLEAN_DIR = r"D:\Malware Dataset\processed_clean"
FINAL_DIR = r"D:\Malware Dataset\processed_final"
os.makedirs(FINAL_DIR, exist_ok=True)

CATEGORIES = ["Backdoor", "Benign", "Exploit", "HackTool", "Hoax",
              "Rootkit", "Trojan", "Virus", "Worm"]

def read_parquet_safe(path: str) -> pl.DataFrame:
    return pl.read_parquet(path, memory_map=False)  # same Windows-safety fix as before

def load_all_categories(prefix: str) -> pd.DataFrame:
    frames = []
    for cat in CATEGORIES:
        path = os.path.join(CLEAN_DIR, f"{prefix}_{cat}_dedup.parquet")
        if os.path.exists(path):
            frames.append(read_parquet_safe(path))
    combined = pl.concat(frames, how="diagonal_relaxed")
    return combined.to_pandas()

# %%
# ============================================================
# STEP 5: Grouped stratified split — BEFORE any resampling
# ============================================================
# StratifiedGroupKFold keeps rows from the same sample_id (capture file)
# entirely on one side of the split, while still balancing class
# proportions across train/test — exactly what you specified.

net_pd = load_all_categories("net")
print(f"Network combined: {net_pd.shape}")

sgkf_net = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf_net.split(
    net_pd, net_pd["label"], groups=net_pd["sample_id"]
))
net_train = net_pd.iloc[train_idx].reset_index(drop=True)
net_test = net_pd.iloc[test_idx].reset_index(drop=True)

# Sanity check — this must print 0. If it doesn't, something's wrong
# with how sample_id was assigned upstream.
overlap = set(net_train["sample_id"]) & set(net_test["sample_id"])
print(f"Network train: {net_train.shape}, test: {net_test.shape}")
print(f"Sample ID overlap between train/test: {len(overlap)} (must be 0)")
print("\nTrain label distribution:")
print(net_train["label"].value_counts())
print("\nTest label distribution:")
print(net_test["label"].value_counts())

# %%
# Same grouped split for Memory — lower leakage risk (rows are closer to
# 1:1 with files) but kept for consistency, as you noted.
mem_pd = load_all_categories("mem")
print(f"Memory combined: {mem_pd.shape}")

sgkf_mem = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
mem_train_idx, mem_test_idx = next(sgkf_mem.split(
    mem_pd, mem_pd["label"], groups=mem_pd["sample_id"]
))
mem_train = mem_pd.iloc[mem_train_idx].reset_index(drop=True)
mem_test = mem_pd.iloc[mem_test_idx].reset_index(drop=True)

mem_overlap = set(mem_train["sample_id"]) & set(mem_test["sample_id"])
print(f"Memory train: {mem_train.shape}, test: {mem_test.shape}")
print(f"Sample ID overlap between train/test: {len(mem_overlap)} (must be 0)")
print("\nTrain label distribution:")
print(mem_train["label"].value_counts())

# %%
# ============================================================
# STEP 6: Log-transform heavy-tailed features
# ============================================================
# log1p handles zeros safely (log1p(0) = 0, unlike log(0) = -inf).
# Applied identically to train and test — this is a stateless, row-wise
# transform, so it can't leak information between the two sets.

LOG_TRANSFORM_COLS = ["duration", "packets_count", "total_payload_bytes",
                       "bytes_rate", "packets_rate"]

def apply_log1p(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            # clip guards against any stray negative sentinel values
            # (e.g. the -1 fill from Step 4's structural-missingness handling)
            df[c] = np.log1p(df[c].clip(lower=0))
    return df

net_train = apply_log1p(net_train, LOG_TRANSFORM_COLS)
net_test = apply_log1p(net_test, LOG_TRANSFORM_COLS)
print("Log-transform applied to:", [c for c in LOG_TRANSFORM_COLS if c in net_train.columns])

# %%
# ============================================================
# STEP 7: Min-Max scale — fit on TRAIN ONLY, apply to both
# ============================================================
# One addition beyond what you specified: the scaler must be FIT only on
# training data, then used to TRANSFORM the test set with those same
# fitted min/max values — never re-fit on test. Fitting on the full
# dataset (train+test together) would leak test-set distribution
# information into training, undermining the whole point of Step 5's split.

# EDIT this list if your actual column names differ — these are
# identifier/non-feature columns to exclude from scaling.
NON_FEATURE_COLS = ["label", "sample_id", "flow_id", "timestamp",
                     "src_ip", "dst_ip", "protocol"]

# FIX (root cause of Training_XGBoost.ipynb's Optuna trial-0 crash):
# NON_FEATURE_COLS above only ever excluded these columns from *scaling*.
# It never dropped them from net_train/net_test, so flow_id/timestamp/
# src_ip/dst_ip/protocol (and any other leftover non-numeric column from
# the ignore_errors=True CSV load) survived into the saved Parquet as
# strings, which XGBoost cannot consume. None of these are model
# features -- drop the true identifier columns outright, and drop any
# other non-numeric column that isn't the label itself.
identifier_cols_present = [c for c in NON_FEATURE_COLS if c in net_train.columns and c != "label" and c != "sample_id"]
other_non_numeric = [c for c in net_train.columns
                      if c not in ("label", "sample_id") and c not in identifier_cols_present
                      and not pd.api.types.is_numeric_dtype(net_train[c])]
cols_to_drop = identifier_cols_present + other_non_numeric
if cols_to_drop:
    print(f"Dropping {len(cols_to_drop)} non-feature/non-numeric column(s) entirely: {cols_to_drop}")
    net_train = net_train.drop(columns=[c for c in cols_to_drop if c in net_train.columns])
    net_test = net_test.drop(columns=[c for c in cols_to_drop if c in net_test.columns])


numeric_cols = [c for c in net_train.columns
                if c not in NON_FEATURE_COLS and pd.api.types.is_numeric_dtype(net_train[c])]

scaler = MinMaxScaler()
net_train[numeric_cols] = scaler.fit_transform(net_train[numeric_cols])
net_test[numeric_cols] = scaler.transform(net_test[numeric_cols])
print(f"Scaled {len(numeric_cols)} numeric columns (fit on train, applied to test)")

# %%
# ============================================================
# STEP 8: Prune highly correlated features (|r| > 0.9)
# ============================================================
# Computed on TRAIN ONLY, then the same drop list is applied to test —
# consistent with never letting test data influence any decision.

corr_matrix = net_train[numeric_cols].corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.9)]

print(f"Dropping {len(to_drop)} highly correlated columns (|r| > 0.9):")
print(to_drop)

net_train = net_train.drop(columns=to_drop)
net_test = net_test.drop(columns=to_drop)
numeric_cols = [c for c in numeric_cols if c not in to_drop]
print(f"Remaining feature columns: {len(numeric_cols)}")

# %%
# ============================================================
# STEP 9: Network imbalance handling — moderate (~5:1), light touch
# ============================================================
# Default here is class-weighting rather than SMOTE: it corrects the
# imbalance in training WITHOUT growing the dataset — relevant given
# your earlier memory constraints. XGBoost consumes these weights
# directly via sample_weight in .fit().

y_train_net = net_train["label"]
net_sample_weights = compute_sample_weight(class_weight="balanced", y=y_train_net)
print("Computed balanced sample weights for Network training set.")
print("Pass these to XGBoost as: model.fit(X_train, y_train, sample_weight=net_sample_weights)")

# --- Alternative: light multiclass SMOTE instead of weighting ---
# Uncomment this block INSTEAD of using net_sample_weights if you'd
# rather oversample. Note this DOES increase net_train's row count.
#
# from imblearn.over_sampling import SMOTE
# feature_cols_net = [c for c in numeric_cols]
# smote_net = SMOTE(random_state=42, k_neighbors=5)
# X_res, y_res = smote_net.fit_resample(net_train[feature_cols_net], y_train_net)
# print(f"After SMOTE: {X_res.shape[0]} rows (was {net_train.shape[0]})")

# %%
# ============================================================
# STEP 10: Memory imbalance handling — cluster-based SMOTE for
# Exploit specifically, light weighting for the other 8 categories
# ============================================================

def smote_within_cluster(X_cluster: np.ndarray, n_synthetic: int,
                          k_neighbors: int = 3, random_state: int = 42) -> np.ndarray:
    """Manual SMOTE interpolation within one cluster: for each new point,
    pick a real point and one of its k nearest neighbors (within the same
    cluster), then interpolate along the line between them —
    x_syn = x_i + lambda * (x_neighbor - x_i), lambda in [0,1].
    This is the same formula used in Siagian et al.'s SMOTE description,
    applied here per-cluster instead of across the whole minority class."""
    rng = np.random.default_rng(random_state)
    n_samples = X_cluster.shape[0]
    k = min(k_neighbors, n_samples - 1)

    if k < 1:
        # Too few points in this cluster to interpolate meaningfully —
        # fall back to duplication rather than fabricating from noise.
        idx = rng.integers(0, n_samples, size=n_synthetic)
        return X_cluster[idx]

    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_cluster)
    _, neighbor_idx = nn.kneighbors(X_cluster)

    synthetic = np.zeros((n_synthetic, X_cluster.shape[1]))
    for s in range(n_synthetic):
        i = rng.integers(0, n_samples)
        neighbor_choices = neighbor_idx[i][1:]  # exclude the point itself
        j = rng.choice(neighbor_choices)
        lam = rng.random()
        synthetic[s] = X_cluster[i] + lam * (X_cluster[j] - X_cluster[i])
    return synthetic

# %%
# ============================================================
# STEP 10.0 (addition): Handle remaining missing values + scale
# Memory features before clustering
# ============================================================
# Steps 1-4 only checked/handled info.winBuild (100% missing, dropped).
# Other Memory columns may have partial missingness across the 9
# categories that was never checked — and KMeans errors out on any NaN.
#
# Separately: KMeans computes Euclidean distances between points, so it
# has the exact same sensitivity to missing values AND mismatched feature
# scales that SMOTE does. Step 7 already scaled Network features for this
# reason — this does the equivalent for Memory before clustering, since
# the original step list didn't call for it but the clustering step
# depends on it to produce meaningful clusters at all.

mem_numeric_cols = [c for c in mem_train.columns
                    if c not in NON_FEATURE_COLS and pd.api.types.is_numeric_dtype(mem_train[c])]

null_counts = mem_train[mem_numeric_cols].isnull().sum()
nulls_found = null_counts[null_counts > 0].sort_values(ascending=False)
print("Memory training columns with missing values (not caught by Steps 1-4):")
print(nulls_found if len(nulls_found) else "  none found")

# Median imputation — fit on train, apply the SAME medians to test.
# Reasonable default here since, unlike Network's delta_start/
# handshake_duration, Step 4 never investigated whether Memory's
# missingness is structural — if the printed list above looks
# concentrated in specific categories rather than spread evenly,
# treat it the same way Step 4 did (sentinel + flag) instead of
# median-filling it blindly.
train_medians = mem_train[mem_numeric_cols].median()
mem_train[mem_numeric_cols] = mem_train[mem_numeric_cols].fillna(train_medians)
mem_test[mem_numeric_cols] = mem_test[mem_numeric_cols].fillna(train_medians)

# Scale — fit on train only, same leakage-avoidance principle as Step 7
mem_scaler = MinMaxScaler()
mem_train[mem_numeric_cols] = mem_scaler.fit_transform(mem_train[mem_numeric_cols])
mem_test[mem_numeric_cols] = mem_scaler.transform(mem_test[mem_numeric_cols])

print(f"Imputed missing values (median) and scaled {len(mem_numeric_cols)} Memory feature columns.")

# %%
EXPLOIT_LABEL = "Exploit"
mem_feature_cols = mem_numeric_cols  # clean, NaN-free, scaled columns only

exploit_train = mem_train[mem_train["label"] == EXPLOIT_LABEL].reset_index(drop=True)
other_train = mem_train[mem_train["label"] != EXPLOIT_LABEL].reset_index(drop=True)

print(f"Exploit real samples in training set: {len(exploit_train)}")
print("Other category counts:")
print(other_train["label"].value_counts())

# %%
# Step 10.1 — choose K for Exploit clustering via Silhouette score
X_exploit = exploit_train[mem_feature_cols].values

best_k, best_score = None, -1
max_k_to_try = min(4, len(X_exploit) - 1)  # small K expected given ~100 samples
for k in range(2, max_k_to_try + 1):
    cluster_labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_exploit)
    score = silhouette_score(X_exploit, cluster_labels)
    print(f"  K={k}: silhouette = {score:.3f}")
    if score > best_score:
        best_k, best_score = k, score

print(f"Chosen K = {best_k} (silhouette = {best_score:.3f})")
exploit_cluster_labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X_exploit)

# %%
# Step 10.2 — generate synthetic samples per cluster, proportional to size
# EDIT this target based on what balance you actually want. A reasonable
# default: match the smallest of the OTHER 8 categories, so Exploit stops
# being the extreme outlier without over-inflating it relative to real data.
target_total_exploit = int(other_train["label"].value_counts().min())
print(f"Target total Exploit count after augmentation: {target_total_exploit}")

n_needed = max(0, target_total_exploit - len(X_exploit))
print(f"Synthetic samples needed: {n_needed}")

synthetic_rows = []
if n_needed > 0:
    for c in range(best_k):
        cluster_mask = exploit_cluster_labels == c
        cluster_data = X_exploit[cluster_mask]
        cluster_proportion = cluster_mask.sum() / len(X_exploit)
        n_for_cluster = int(round(n_needed * cluster_proportion))

        if n_for_cluster == 0:
            continue

        synth = smote_within_cluster(cluster_data, n_for_cluster, k_neighbors=3)
        synth_df = pd.DataFrame(synth, columns=mem_feature_cols)
        synth_df["label"] = EXPLOIT_LABEL
        synth_df["sample_id"] = [f"Exploit_synthetic_{c}_{i}" for i in range(n_for_cluster)]
        synth_df["is_synthetic"] = 1
        synthetic_rows.append(synth_df)
        print(f"  Cluster {c}: {cluster_mask.sum()} real -> {n_for_cluster} synthetic generated")

exploit_train["is_synthetic"] = 0
if synthetic_rows:
    exploit_augmented = pd.concat([exploit_train] + synthetic_rows, ignore_index=True)
else:
    exploit_augmented = exploit_train
print(f"Exploit training set after augmentation: {len(exploit_augmented)} rows "
      f"({exploit_augmented['is_synthetic'].sum()} synthetic)")

# %%
# Step 10.3 — recombine, then apply light weighting for the remaining
# (now much less severe) imbalance across all 9 memory categories
other_train["is_synthetic"] = 0
mem_train_final = pd.concat([exploit_augmented, other_train], ignore_index=True)

y_train_mem = mem_train_final["label"]
mem_sample_weights = compute_sample_weight(class_weight="balanced", y=y_train_mem)
print("Computed balanced sample weights for Memory training set.")
print("Final Memory training label distribution:")
print(mem_train_final["label"].value_counts())

# %%
# ============================================================
# Save everything needed for the next stage (model training)
# ============================================================
net_train.to_parquet(os.path.join(FINAL_DIR, "net_train.parquet"))
net_test.to_parquet(os.path.join(FINAL_DIR, "net_test.parquet"))
mem_train_final.to_parquet(os.path.join(FINAL_DIR, "mem_train.parquet"))
mem_test.to_parquet(os.path.join(FINAL_DIR, "mem_test.parquet"))

np.save(os.path.join(FINAL_DIR, "net_sample_weights.npy"), net_sample_weights)
np.save(os.path.join(FINAL_DIR, "mem_sample_weights.npy"), mem_sample_weights)

print("\nSteps 5-10 complete. Final train/test sets and sample weights saved to:", FINAL_DIR)
print("IMPORTANT: mem_train_final includes an 'is_synthetic' column — carry this "
      "through to your results/discussion for Exploit, per your own caveat about "
      "lower-confidence conclusions on that class.")